# Practice Session 04: Weighted Networks


In this session we will reconstruct the World Trade Web by using [NetworkX](https://networkx.github.io/). This practice is inspired by a paper analysing the Topology of the World Trade Web (https://arxiv.org/abs/cond-mat/0301015).

The datasets we will use comes from OECD's [quarterly international trade statistics](https://stats.oecd.org/Index.aspx?DataSetCode=QITS#) dataset, and these are your input files:

* ``trade-2012-flows.csv`` international trade in 2012
* ``trade-2022-flows.csv`` international trade in 2022
* ``trade-countries.csv`` list of countries and territories

<font size="-1" color="gray">(Remove this cell when delivering.)</font>

# 1. Read mapping of codes to country names

Read into variable ``id2name`` the file containing the list of countries and territories.

This is a comma-separated file with two columns: *code*, which is a letter code (e.g. ``PRT``),  and *name* (e.g., ``Portugal``).
<font size="-1" color="gray">(Remove this cell when delivering.)</font>

In [ ]:
import csv
import io
import networkx as nx
import matplotlib.pyplot as plt
import math
import pandas as pd

In [ ]:
# Leave this cell as-is

INPUT_NAMES = "trade-countries.csv"
INPUT_2O12 = "trade-2012-flows.csv"
INPUT_2022 = "trade-2022-flows.csv"
YEAR_OLD = 2012
YEAR_NEW = 2022

In [ ]:
id2name = {}

<font size="+1" color="red">Replace this cell with your code to read country names into id2name.</font>

Test the mapping using the following code.

<font size="-1" color="gray">(Remove this cell when delivering.)</font>

In [ ]:
# LEAVE AS-IS, it should print "Switzerland"

print(id2name["CHE"])

# 2. Read flows data into two graphs

## 2.1. Read amount traded

The files `INPUT_2012` and `INPUT_2022` contain three columns:

* from (country from which the goods or services are exported)
* to (country that receives the goods or services)
* amount (total amount in USD)

Read the graphs as **directed graphs** into variables `gOld` and `gNew` respectively.

By default the columns read by a DictReader are string, but you can convert the column containing the amount exported to a floating point and divide the `amount` by one million and round to the nearest integer, so the weights will be expressed in millions of dollars.

**Important:** create only the nodes corresponding to countries listed in ``INPUT_NAMES`` (keys in ``id2name``)

<font size="-1" color="gray">(Remove this cell when delivering.)</font>

<font size="+1" color="red">Replace this cell with your code to read the two graphs: ``gOld`` and ``gNew``.</font> **2p**

The following should print:

```
Switzerland
The 2012 graph contains 33 nodes and 1049 edges with a total weight of 6,616,535 million USD
The 2022 graph contains 33 nodes and 823 edges with a total weight of 7,564,211 million USD
```

If it prints more than that, perhaps you read some edges having one of the two nodes outside of the list of countries.

<font size="-1" color="gray">(Remove this cell when delivering.)</font>

In [ ]:
# LEAVE AS-IS

print("The {:d} graph contains {:d} nodes and {:d} edges with a total weight of {:,d} million USD".format(YEAR_OLD, gOld.number_of_nodes(),gOld.size(),sum(d.get("weight", 0) for u, v, d in gOld.edges(data=True))))
print("The {:d} graph contains {:d} nodes and {:d} edges with a total weight of {:,d} million USD".format(YEAR_NEW, gNew.number_of_nodes(),gNew.size(),sum(d.get("weight", 0) for u, v, d in gNew.edges(data=True))))

The following should print:

```
In 2012, ESP (Spain) exported to SVK (Slovak Republic) goods and services for 777 USD millions
by 2022, it exported 1,548 USD millions

In 2012, ESP (Spain) exported to ITA (Italy) goods and services for 21,601 USD millions
by 2022, it exported 32,225 USD millions

In 2012, USA (United States) exported to SVK (Slovak Republic) goods and services for 333 USD millions
by 2022, it exported 376 USD millions

In 2012, USA (United States) exported to ITA (Italy) goods and services for 15,972 USD millions
by 2022, it exported 27,411 USD millions
```

<font size="-1" color="gray">(Remove this cell when delivering.)</font>

In [ ]:
# LEAVE AS-IS

for exporter in ["ESP", "USA"]:
    for importer in ["SVK", "ITA"]:
        print("In {:d}, {:s} ({:s}) exported to {:s} ({:s}) goods and services for {:,d} USD millions".format(
            YEAR_OLD, exporter, id2name[exporter], importer, id2name[importer],
               gOld.get_edge_data(exporter, importer)["weight"]) )

        print("by {:d}, it exported {:,d} USD millions".format(YEAR_NEW, gNew.get_edge_data(exporter, importer)["weight"]))
        print("")

## 2.2. Compute totals

Next, compute the total exports of each country into two dictionaries: `exportsOld` and `exportsNew`, with keys being country codes, and values being total exports. Do the same with total imports, into variables `importsOld` and `importsNew`.

To avoid code duplication, create a function `sum_weights(graph, direction, weight_key)` to avoid duplication. This function should receive a *graph*, a *direction* ("in" or "out"), and a *weight_key*, defaulting to "weight".

    
<font size="-1" color="gray">(Remove this cell when delivering.)</font>

<font size="+1" color="red">Replace this cell with your code to compute totals into dictionaries exportsOld, exportsNew, importsOld, importsNew. Create a function *sum_weights* to avoid code duplication.</font>

The following should print:

```
Estonia exported 10,082 USD Million in 2012 and 14,312 USD Million in 2022 (growth 1.4x),
 and imported 11,401 USD Million in 2012 and 10,463 USD Million in 2022 (growth 0.9x).

Spain exported 214,025 USD Million in 2012 and 313,679 USD Million in 2022 (growth 1.5x),
 and imported 208,224 USD Million in 2012 and 234,182 USD Million in 2022 (growth 1.1x).

Türkiye exported 66,536 USD Million in 2012 and 133,088 USD Million in 2022 (growth 2.0x),
 and imported 112,203 USD Million in 2012 and 110,220 USD Million in 2022 (growth 1.0x).
```

<font size="-1" color="gray">(Remove this cell when delivering.)</font>

In [ ]:
# LEAVE AS-IS

for country in ['EST', 'ESP', 'TUR']:
    print("{:s} exported {:,d} USD Million in {:d} and {:,d} USD Million in {:d} (growth {:.1f}x),".format(
        id2name[country],
        exportsOld[country], YEAR_OLD,
        exportsNew[country], YEAR_NEW,
        exportsNew[country]/exportsOld[country]
    ))
    print(" and imported {:,d} USD Million in {:d} and {:,d} USD Million in {:d} (growth {:.1f}x).".format(
        importsOld[country], YEAR_OLD,
        importsNew[country], YEAR_NEW,
        importsNew[country]/importsOld[country]
    ))
    print()

# 3. Draw the net trade increase

Draw a directed graph using NetworkX with the following characteristics:

* **node_size** proportional to **net exports in 2022**.
* **Directed links** in the direction of **net trade increase**. Width proportional to net trade increase.
* Use a **spring model** layout. Fix k at 100 to avoid overlapping
* Make sure arrow heads are visible.

Tips:
* Use a small multiplier between 0.001 and 0.005 to get reasonable node sizes.
* Use a small multiplier between0.00001 and 0.00005 to get reasonable link widths
* Fix k at 100 to avoid overlap
* min_target_margin is a list. The min_target_margin needs to be the corresponding target node's radius:  
sqrt(node_size/π)

​

<font size="+1" color="red">Replace this cell with your code to plot the graph.</font>

<font size="+1" color="red">Replace this cell with:

* A. Look at Italy (ITA) and mention any anomaly.

* B. Compare with the data in the csv Files and identify the origin of the anomaly.

* C. Mention whether this happens with other countries, if so, name 2.

</font>

# 4. Comparison of export and import scores

Using **2012** data:

* Create a **loglog scatterplot** with Imports on the x axis and Exports in the Y axis
* Include only countries whoose **imports and exports are above 10E5 Million**
* Include a **diagonal line** for 0 trade balance.
* Include **axis labels**.

<font size="-1" color="gray">(Remove this cell when delivering.)</font>

<font size="+1" color="red">Replace this cell with your function *plot_scatter*. **Do not forget to filter out cases where values of *a* or *b* are 0.**</font>

<font size="+1" color="red">Which are the two countries with most imports? Name 2 countries that export more than they import.</font>

# DELIVER (individually)

A .zip file containing:

* This notebook.


<font size="+2" color="#003300">I hereby declare that, except for the code provided by the course instructors, all of my code, report, and figures were produced by myself.</font>